# Exercise 1 Prompt Chaining for Customer Support

**Tools used:** Google Colab, Python, Google AI Studio, the Gemini API, and the `google-genai` Python SDK.

**Goal:** Build a customer-support prompt chain in which every response becomes part of the next prompt.

**Flow:** classify the issue -> identify missing information -> propose the next action -> apply an escalation rule.

## Test and improvement

**Initial prompt:** `Write a response to a late order.`

The initial prompt was too vague. It did not classify the problem, identify missing information, define a safe response, or specify when the issue should be escalated. The revised version uses four linked prompts with clear roles, formats, tone rules, and limits.

## Before running

In Colab, select the key icon on the left. Add a secret named `GEMINI_API_KEY`, paste your Gemini API key as the value, and enable notebook access. Never type the key directly into a code cell.

In [3]:
!pip -q install -U google-genai

from google import genai
from google.colab import userdata

try:
    api_key = userdata.get("GEMINI_API_KEY")
    if not api_key:
        raise ValueError
except Exception:
    raise ValueError(
        "GEMINI_API_KEY was not found. Add it under the key icon in Colab "
        "and turn on notebook access."
    )

client = genai.Client(api_key=api_key)
MODEL = "gemini-3.5-flash-lite"
print("Gemini client is ready. The API key was loaded from Colab Secrets.")

Gemini client is ready. The API key was loaded from Colab Secrets.


## Exact prompts

### Step 1

```text
You are a customer-support triage assistant. Read the customer message below.
Return exactly two lines:
Category: one of late delivery, damaged item, billing, other
Evidence: a short phrase from the message
Do not invent facts or promise a refund.

Customer message: {customer_message}
```

### Step 2

```text
You are a customer-support assistant. Use the classification and original message below.
Return exactly two lines:
Known: concise facts already supplied
Ask: one question for the most important missing detail
Do not request payment details or repeat information already provided.

Classification:
{classification}

Customer message:
{customer_message}
```

### Step 3

```text
You are a customer-support assistant. Use the classification and information check.
Write two sentences. First, acknowledge the issue in a calm and respectful tone.
Second, explain the next action, depending on the missing information.
Do not claim that you checked a real account, guarantee a delivery date, or promise a refund.

Classification:
{classification}

Information check:
{information_check}
```

### Step 4

```text
You are a customer-support supervisor. Use all earlier results.
If the customer reports a missed promised delivery date or a time-sensitive need,
write Escalate: yes. Otherwise, write Escalate: no.
Then write Reason: one short sentence and Final reply: no more than three polite sentences.
Preserve the proposed next action and never claim access to the customer's account.

Classification:
{classification}

Information check:
{information_check}

Proposed response:
{proposal}

Customer message:
{customer_message}
```

In [4]:
PROMPTS = ['You are a customer-support triage assistant. Read the customer message below.\nReturn exactly two lines:\nCategory: one of late delivery, damaged item, billing, other\nEvidence: a short phrase from the message\nDo not invent facts or promise a refund.\n\nCustomer message: {customer_message}', 'You are a customer-support assistant. Use the classification and original message below.\nReturn exactly two lines:\nKnown: concise facts already supplied\nAsk: one question for the most important missing detail\nDo not request payment details or repeat information already provided.\n\nClassification:\n{classification}\n\nCustomer message:\n{customer_message}', 'You are a customer-support assistant. Use the classification and information check.\nWrite two sentences. First, acknowledge the issue in a calm and respectful tone.\nSecond, explain the next action, depending on the missing information.\nDo not claim that you checked a real account, guarantee a delivery date, or promise a refund.\n\nClassification:\n{classification}\n\nInformation check:\n{information_check}', "You are a customer-support supervisor. Use all earlier results.\nIf the customer reports a missed promised delivery date or a time-sensitive need,\nwrite Escalate: yes. Otherwise, write Escalate: no.\nThen write Reason: one short sentence and Final reply: no more than three polite sentences.\nPreserve the proposed next action and never claim access to the customer's account.\n\nClassification:\n{classification}\n\nInformation check:\n{information_check}\n\nProposed response:\n{proposal}\n\nCustomer message:\n{customer_message}"]

customer_message = (
    "My order was promised yesterday, but it has not arrived. "
    "I need it for an event tomorrow."
)

def ask_gemini(prompt):
    response = client.models.generate_content(model=MODEL, contents=prompt)
    return response.text.strip()

context = {"customer_message": customer_message}
keys = ["classification", "information_check", "proposal", "decision"]

for step, (key, template) in enumerate(zip(keys, PROMPTS), start=1):
    prompt = template.format(**context)
    response = ask_gemini(prompt)
    context[key] = response
    print(f"STEP {step} PROMPT\n{prompt}\n")
    print(f"STEP {step} GEMINI OUTPUT\n{response}\n")

print("FINAL CHAIN RESULT\n" + context["decision"])

STEP 1 PROMPT
You are a customer-support triage assistant. Read the customer message below.
Return exactly two lines:
Category: one of late delivery, damaged item, billing, other
Evidence: a short phrase from the message
Do not invent facts or promise a refund.

Customer message: My order was promised yesterday, but it has not arrived. I need it for an event tomorrow.

STEP 1 GEMINI OUTPUT
Category: late delivery
Evidence: promised yesterday, but it has not arrived

STEP 2 PROMPT
You are a customer-support assistant. Use the classification and original message below.
Return exactly two lines:
Known: concise facts already supplied
Ask: one question for the most important missing detail
Do not request payment details or repeat information already provided.

Classification:
Category: late delivery
Evidence: promised yesterday, but it has not arrived

Customer message:
My order was promised yesterday, but it has not arrived. I need it for an event tomorrow.

STEP 2 GEMINI OUTPUT
Known: Ord

## Successful-output check

After running, confirm that all four steps are visible. The result should classify the complaint as a late delivery, ask for the order number, recommend a safe next action, and escalate the case because the promised date passed and the customer has a time-sensitive need.